# Feast Feature Store

1. Feature Views: `driver_efficiency` and `driver_activity`
2. On-demand Feature View: `driver_scoring`
3. Historical feature retrieval
4. Online feature retrieval

In [1]:
import pandas as pd
from datetime import datetime
from feast import FeatureStore
import warnings
warnings.filterwarnings('ignore')

## Setup: Apply & Materialize

In [3]:
store = FeatureStore(repo_path='../feature_store')

# Materialize features into online store (SQLite)
store.materialize_incremental(end_date=datetime.now())
print('Materialization complete')

Materializing 2 feature views to 2026-03-21 10:49:27+00:00 into the sqlite online store.

driver_efficiency from 2026-03-21 10:49:17+00:00 to 2026-03-21 10:49:27+00:00:
driver_activity from 2026-03-21 10:49:17+00:00 to 2026-03-21 10:49:27+00:00:
Materialization complete


## Raw Data Overview

In [4]:
raw = pd.read_parquet('../feature_store/data/driver_stats.parquet')
print(f'Shape: {raw.shape}')
print(f'Drivers: {sorted(raw.driver_id.unique())}')
raw.head(10)

Shape: (1808, 6)
Drivers: [np.int64(1001), np.int64(1002), np.int64(1003), np.int64(1004), np.int64(1005)]


,event_timestamp,driver_id,conv_rate,acc_rate,avg_daily_trips,created
0,2024-10-17 12:07:08.228578+00:00,1001,1.000000,1.000000,1000,2024-10-17 12:07:08.228581
1,2024-10-02 11:00:00+00:00,1005,0.429879,0.194598,582,2024-10-17 11:30:07.072000
2,2024-10-02 12:00:00+00:00,1005,0.230119,0.642878,551,2024-10-17 11:30:07.072000
3,2024-10-02 13:00:00+00:00,1005,0.128600,0.674187,38,2024-10-17 11:30:07.072000
4,2024-10-02 14:00:00+00:00,1005,0.400603,0.473636,583,2024-10-17 11:30:07.072000
5,2024-10-02 15:00:00+00:00,1005,0.803716,0.016832,547,2024-10-17 11:30:07.072000
6,2024-10-02 16:00:00+00:00,1005,0.561917,0.138757,572,2024-10-17 11:30:07.072000
7,2024-10-02 17:00:00+00:00,1005,0.232691,0.102074,385,2024-10-17 11:30:07.072000
8,2024-10-02 18:00:00+00:00,1005,0.976390,0.415044,143,2024-10-17 11:30:07.072000
9,2024-10-02 19:00:00+00:00,1005,0.051284,0.780878,398,2024-10-17 11:30:07.072000


## Feature Views

| Feature View | Type | Features | Description |
|---|---|---|---|
| `driver_efficiency` | Batch | conv_rate, acc_rate | Driver quality metrics |
| `driver_activity` | Batch | avg_daily_trips | Trip volume |
| `driver_scoring` | On-demand | weighted_rating, trip_score, surge_adjusted_score | Real-time composite score |

## 1. Historical Features (Trainin)

In [7]:
entity_df = pd.DataFrame({
    'driver_id': [1001, 1002, 1003, 1004, 1005],
    'event_timestamp': pd.to_datetime([
        '2024-10-17 12:00:00',
        '2024-10-17 11:00:00',
        '2024-10-17 10:00:00',
        '2024-10-17 09:00:00',
        '2024-10-17 08:00:00',
    ], utc=True),
    'surge_multiplier': [1.0, 1.5, 2.0, 1.2, 0.8],
})

print('Entity DataFrame:')
entity_df

Entity DataFrame:


,driver_id,event_timestamp,surge_multiplier
0,1001,2024-10-17 12:00:00+00:00,1.0
1,1002,2024-10-17 11:00:00+00:00,1.5
2,1003,2024-10-17 10:00:00+00:00,2.0
3,1004,2024-10-17 09:00:00+00:00,1.2
4,1005,2024-10-17 08:00:00+00:00,0.8


In [8]:
# Get historical features
training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        'driver_efficiency:conv_rate',
        'driver_efficiency:acc_rate',
        'driver_activity:avg_daily_trips',
        'driver_scoring:weighted_rating',
        'driver_scoring:trip_score',
        'driver_scoring:surge_adjusted_score',
    ],
).to_df()

print('Training DataFrame (historical features):')
training_df

Training DataFrame (historical features):


,driver_id,event_timestamp,surge_multiplier,conv_rate,acc_rate,avg_daily_trips,weighted_rating,trip_score,surge_adjusted_score
0,1005,2024-10-17 08:00:00+00:00,0.8,0.045158,0.448728,723,0.206586,1.360325,1.088260
1,1004,2024-10-17 09:00:00+00:00,1.2,0.683725,0.951837,238,0.790970,4.331719,5.198063
2,1003,2024-10-17 10:00:00+00:00,2.0,0.934453,0.695821,763,0.839000,5.569758,11.139517
3,1001,2024-10-17 12:00:00+00:00,1.0,0.295067,0.257097,456,0.279879,1.714169,1.714169
4,1002,2024-10-17 11:00:00+00:00,1.5,0.134940,0.982010,696,0.473768,3.101659,4.652489


### On-demand features

`driver_scoring` view:

- `weighted_rating` = conv_rate × 0.6 + acc_rate × 0.4
- `trip_score` = weighted_rating × log(1 + avg_daily_trips)
- `surge_adjusted_score` = trip_score × surge_multiplier

`surge_multiplier` - from request

## 2. Online Features (Inference)

In [9]:
online_features = store.get_online_features(
    features=[
        'driver_efficiency:conv_rate',
        'driver_efficiency:acc_rate',
        'driver_activity:avg_daily_trips',
        'driver_scoring:weighted_rating',
        'driver_scoring:trip_score',
        'driver_scoring:surge_adjusted_score',
    ],
    entity_rows=[
        {'driver_id': 1001, 'surge_multiplier': 1.5},
        {'driver_id': 1002, 'surge_multiplier': 1.0},
        {'driver_id': 1005, 'surge_multiplier': 2.0},
    ],
).to_dict()

print('Online features (for inference):')
pd.DataFrame(online_features)

Online features (for inference):


,driver_id,acc_rate,conv_rate,avg_daily_trips,weighted_rating,trip_score,surge_adjusted_score
0,1001,1.000000,1.000000,1000,1.000000,6.908755,10.363132
1,1002,0.982010,0.134940,696,0.473768,3.101659,3.101659
2,1005,0.752123,0.530015,228,0.618858,3.362703,6.725405


## 3. Feature View Metadata

In [10]:
for fv in store.list_feature_views():
    print(f'Feature View: {fv.name}')
    print(f'  Entities: {[e.name for e in fv.entity_columns]}')
    print(f'  Features: {[f.name for f in fv.features]}')
    print(f'  TTL: {fv.ttl}')
    print(f'  Tags: {fv.tags}')
    print()

print('On-demand Feature Views:')
for odfv in store.list_on_demand_feature_views():
    print(f'  {odfv.name}: {[f.name for f in odfv.features]}')

Feature View: driver_efficiency
  Entities: ['driver_id']
  Features: ['conv_rate', 'acc_rate']
  TTL: 1095 days, 0:00:00
  Tags: {'domain': 'efficiency', 'team': 'driver_performance'}

Feature View: driver_activity
  Entities: ['driver_id']
  Features: ['avg_daily_trips']
  TTL: 1095 days, 0:00:00
  Tags: {'domain': 'activity', 'team': 'driver_performance'}

On-demand Feature Views:
  driver_scoring: ['weighted_rating', 'trip_score', 'surge_adjusted_score']
